In [16]:
import json
from pathlib import Path
from typing import Dict, Any
import torch

from faster_whisper import WhisperModel

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForAudioClassification,
    pipeline,
    Wav2Vec2FeatureExtractor,
    AutoProcessor,
    Wav2Vec2ForSequenceClassification
)
import librosa
print("Готово")

Готово


In [11]:
!pip install faster-whisper --quiet
!pip install faster-whisper


from faster_whisper import WhisperModel

Defaulting to user installation because normal site-packages is not writeable


In [18]:
# ASR (faster-whisper small)
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float32")
# Text emotion
text_emotion_pipeline = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=0 if device=="cuda" else -1
)
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

voice_model_name = "superb/wav2vec2-base-superb-er"

#feature extractor
voice_processor = AutoFeatureExtractor.from_pretrained(voice_model_name)


voice_model = AutoModelForAudioClassification.from_pretrained(voice_model_name).to(device)

# Метки эмоций
voice_labels = voice_model.config.id2label

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\configuration_utils.py:364: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of the model check

In [19]:
class SpeechToTextEmotion:
    def __init__(self, asr_model, text_emotion_pipeline, voice_model, voice_processor, voice_labels, device="cuda"):
        self.device = device
        self.asr_model = asr_model
        self.text_emotion = text_emotion_pipeline
        self.voice_model = voice_model
        self.voice_processor = voice_processor
        self.voice_labels = voice_labels

    def transcribe(self, audio_path: str) -> str:
        # faster-whisper возвращает генератор сегментов + инфо
        segments, _ = self.asr_model.transcribe(audio_path)
        text = " ".join([seg.text for seg in segments])
        return text.strip()

    def analyze_text_emotion(self, text: str) -> Dict[str, Any]:
        preds = self.text_emotion(text, top_k=1)
        # pipeline возвращает список списков
        pred = preds[0][0] if isinstance(preds[0], list) else preds[0]
        return {"label": pred["label"], "conf": float(pred["score"])}

    def analyze_voice_emotion(self, audio_path: str) -> Dict[str, Any]:
        speech, sr = librosa.load(audio_path, sr=16000)

        inputs = self.voice_processor(
            speech,
            sampling_rate=sr,
            return_tensors="pt"
        )

        with torch.no_grad():
            logits = self.voice_model(inputs["input_values"].to(self.device)).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

        label_id = int(probs.argmax())
        return {
            "label": self.voice_labels[label_id],
            "conf": float(probs[label_id])
        }

    def fuse_emotions(self, voice: Dict[str, Any], text: Dict[str, Any]) -> str:
        if text["conf"] > 0.7:
            return text["label"]
        return voice["label"]

    def process(self, audio_path: str) -> Dict[str, Any]:
        text = self.transcribe(audio_path)
        emo_text = self.analyze_text_emotion(text)
        emo_voice = self.analyze_voice_emotion(audio_path)
        final = self.fuse_emotions(emo_voice, emo_text)

        return {
            "Text": text,
            "Emotion": {
                "voice": emo_voice,
                "text": emo_text,
                "final": final
            }
        }


print("Готово")

Готово


In [20]:
speech_to_text_engine = SpeechToTextEmotion(
    asr_model=asr_model,
    text_emotion_pipeline=text_emotion_pipeline,
    voice_model=voice_model,
    voice_processor=voice_processor,
    voice_labels=voice_labels,
    device=device
)